## Imports

In [37]:
import pandas as pd
import numpy as np
import datetime
import requests
from tqdm import tqdm
from bs4 import BeautifulSoup

## Obter lista de ativos

In [2]:
ativos_ibov = pd.read_excel('refined/ativos_ibov.xlsx')
print(f"Quantidade de linhas: {ativos_ibov.shape[0]}")
ativos_ibov.head()

Quantidade de linhas: 85


,ticker,empresa,tipo_acao,complemento_tipo,segm_gov,qtd_teorica,pct_part,setor,subsetor,segmento
0,ALOS3,ALLOS,ON,NaN,NM,466135796,0.548,Financeiro,Exploração de Imóveis,Exploração de Imóveis
1,ABEV3,AMBEV S/A,ON,NaN,NaN,4273841357,2.495,Consumo não Cíclico,Bebidas,Cervejas e Refrigerantes
2,ASAI3,ASSAI,ON,NaN,NM,1338601375,0.420,Consumo não Cíclico,Comércio e Distribuição,Alimentos
3,AURE3,AUREN,ON,NaN,NM,318338365,0.141,Utilidade Pública,Energia Elétrica,Energia Elétrica
4,AXIA3,AXIA ENERGIA,ON,ATZ,N1,1945846761,4.401,Utilidade Pública,Energia Elétrica,Energia Elétrica


In [5]:
lista_ativos = list(ativos_ibov['ticker'].unique())

## Teste scraping

In [33]:
def get_stock_value(stock):
    url = f'https://www.google.com/finance/quote/{stock}:BVMF?hl=pt'

    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')
    date = datetime.datetime.now()
    price = soup.find('div', {'class':'YMlKec fxKbKc'}).text.encode('utf-8').decode('utf-8').replace('\xa0', ' ').strip()
    # print(price)

    df = pd.DataFrame(data = {'date':[date], 
                              'codigo':[stock],
                              'price':[price]})
    
    df['date'] = pd.to_datetime(df['date']).dt.round('min')
    df['price'] = df['price'].apply(lambda x: x.replace('R$ ', '').replace(',', '.')).astype(float)
    return df

In [40]:
def get_table_stocks(lista_ativos):

    df_precos = pd.DataFrame()

    for ativo in tqdm(lista_ativos):
        try:
            tmp = get_stock_value(ativo)
            df_precos = pd.concat([df_precos, tmp])
        except Exception as e:
            print(e)

    df_precos = df_precos.reset_index(drop = True)

    filename = str(datetime.datetime.now())[:16].replace(' ', '-').replace(':', '-')
    df_precos.to_csv(f'scraped/file-{filename}.csv')

In [41]:
get_table_stocks(lista_ativos)

100%|██████████| 85/85 [01:15<00:00,  1.13it/s]
